# SemanticChunker

`SemanticChunker`는 **의미론적 유사성**을 기반으로 텍스트를 분할합니다.

## 주요 특징

- **의미 기반 분할**: 문장 간 임베딩 유사도를 계산하여 의미적으로 관련된 문장을 함께 그룹화
- **문맥 보존**: 의미가 바뀌는 지점에서만 분할하여 자연스러운 청크 생성
- **동적 청크 크기**: 고정 크기가 아닌 의미 단위로 분할되어 청크 크기가 가변적

## 동작 원리

1. **문장 분리**: 텍스트를 문장 단위로 분리
2. **임베딩 생성**: 각 문장의 임베딩 벡터 계산
3. **유사도 계산**: 인접 문장 간의 임베딩 거리 측정
4. **분할 지점 결정**: 유사도 차이가 큰 지점(breakpoint)에서 분할

## Breakpoint 결정 방식

- **Percentile**: 상위 N% 유사도 차이 지점에서 분할
- **Standard Deviation**: 표준편차 기준으로 이상치 지점에서 분할
- **Interquartile**: 사분위수 범위(IQR) 기준으로 분할

## 사용 시나리오

- ✅ **문맥 보존 중요**: 의미 단위를 끊지 않고 청크를 생성해야 할 때
- ✅ **RAG 품질 향상**: 검색 품질을 높이기 위해 의미적으로 일관된 청크가 필요할 때
- ✅ **Q&A 시스템**: 질문에 대한 답변이 하나의 청크 안에 완전히 포함되어야 할 때

**주의**: OpenAI API 키가 필요합니다 (임베딩 생성).


# 04. SemanticChunker — 의미 기반 분할

## 학습 목표
- `SemanticChunker`의 동작 원리(문장 임베딩 → 유사도 계산 → 분할점 탐색)를 이해해요
- Breakpoint 결정 방식 세 가지(Percentile, Standard Deviation, Interquartile)를 비교해요
- `RecursiveCharacterTextSplitter`와 품질 차이를 확인해요

## 사전 지식
- 03-TokenTextSplitter 완료
- 임베딩(Embedding) 개념 기본 이해
- OpenAI API 키 설정 완료
- `pip install langchain-experimental langchain-openai` 설치

---

> **크기 기반 분할의 한계**: 앞선 방식들은 문자/토큰 **크기**로만 분할해서 의미가 끊길 수 있어요. `SemanticChunker`는 **임베딩 유사도**를 계산해서 의미가 바뀌는 지점에서만 분할해요.

```mermaid
flowchart LR
    S1[문장 1]:::process --> E1[임베딩]:::process
    S2[문장 2]:::process --> E2[임베딩]:::process
    S3[문장 3]:::process --> E3[임베딩]:::process
    E1 --> SIM1{유사도<br/>높음}:::output
    E2 --> SIM1
    E2 --> SIM2{유사도<br/>낮음<br/>분할!}:::error
    E3 --> SIM2

    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
```


## 환경 설정 및 데이터 로드


In [ ]:
# API 키를 환경변수로 관리하기 위한 설정
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()

# 샘플 데이터 파일을 읽어옵니다.
with open("./data/appendix-keywords.txt", encoding="utf-8") as f:
    file = f.read()

print(f"전체 텍스트 길이: {len(file)} 문자")
print(f"\n처음 300자:")
print(file[:300])


## 1. 기본 SemanticChunker

OpenAI 임베딩을 사용하여 의미 기반으로 텍스트를 분할합니다.


In [ ]:
# ============================================================
# TODO: SemanticChunker로 텍스트를 의미 기반으로 분할해보세요
# 힌트: SemanticChunker(OpenAIEmbeddings()) → text_splitter.split_text(file)
# 예상 결과: 크기 기반보다 적은 수의 청크가 생성됩니다 (의미 단위로 묶이기 때문)
# ============================================================

from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai.embeddings import OpenAIEmbeddings

# 1단계: OpenAI 임베딩 기반 SemanticChunker 초기화
# - OpenAIEmbeddings(): 각 문장을 벡터로 변환하는 임베딩 모델
# - 인접 문장 간 코사인 유사도를 계산하여 의미 변화 지점 탐색

# 2단계: 텍스트 분할 (API 호출 발생)


## 2. Percentile 방식

백분위수를 기준으로 상위 N%의 유사도 차이 지점에서 분할합니다.


In [ ]:
# ============================================================
# TODO: breakpoint_threshold_type="percentile"로 분할 임계값을 조정해보세요
# 힌트: SemanticChunker(OpenAIEmbeddings(), breakpoint_threshold_type="percentile", breakpoint_threshold_amount=70)
# 예상 결과: 기본 설정(약 6개)보다 더 많은 청크가 생성됩니다
# ============================================================

# 1단계: Percentile 방식으로 SemanticChunker 설정
# - breakpoint_threshold_amount=70: 유사도 차이가 상위 30%(=100-70)에 해당하는 지점에서 분할
# - 값이 낮을수록 더 많은 지점에서 분할됨


## 3. Standard Deviation 방식

표준편차를 기준으로 이상치 지점에서 분할합니다.


In [ ]:
# Standard Deviation 방식


## 핵심 정리 및 마무리

### Breakpoint 방식 비교

| 방식 | 설명 | 적합한 상황 |
|------|------|-----------|
| `percentile` | 상위 N% 차이 지점에서 분할 | 분할 개수를 대략 조정하고 싶을 때 |
| `standard_deviation` | 통계적으로 명확한 변화 지점 | 의미 변화가 명확한 문서 |
| `interquartile` | 이상치에 덜 민감한 안정적 분할 | 문장 유사도 분포가 고르지 않을 때 |

### 장단점 비교

| 항목 | 크기 기반 (Recursive) | 의미 기반 (Semantic) |
|------|:--------------------:|:-------------------:|
| 속도 | 빠름 | 느림 (임베딩 API 호출) |
| 비용 | 없음 | OpenAI API 비용 발생 |
| 청크 크기 | 균일 | 가변적 |
| 문맥 보존 | 보통 | 우수 |
| RAG 검색 품질 | 보통 | 우수 |

### 실무 팁
> SemanticChunker는 임베딩 API를 호출하기 때문에 비용이 발생해요. 대용량 문서보다는 **품질이 중요한 핵심 문서**에 선별적으로 사용하는 게 효율적이에요.

---

## 다음 예고

일반 텍스트 외에 특수한 형식의 분할 방법을 배워볼게요.

- **05-CodeSplitter**: 프로그래밍 언어 문법을 이해하는 코드 분할
- **06-MarkdownHeaderTextSplitter**: 마크다운 헤더 구조를 보존하는 분할
